In [1]:
%run training_setup.ipynb

c:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier
4172 1044 624
torch.Size([32, 1, 224, 224])
torch.Size([32])
tensor([0, 1, 1, 0, 0, 1, 0, 0, 0, 1])


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_dataset.transform = train_transform
val_dataset.transform   = val_test_transform
test_dataset.transform  = val_test_transform

In [6]:
import torch.nn as nn
from torchvision import models

model = models.efficientnet_b0(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

model = model.to(device)

c:\Users\Nourhan Yehia\.conda\envs\cnn_xray\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Nourhan Yehia\.conda\envs\cnn_xray\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:

for param in model.parameters():
    param.requires_grad = False


for param in model.classifier.parameters():
    param.requires_grad = True

In [8]:
class_counts = train_df["class"].value_counts()

n_normal = class_counts["NORMAL"]
n_pneu   = class_counts["PNEUMONIA"]

total = n_normal + n_pneu

w_normal = total / (2 * n_normal)
w_pneu   = total / (2 * n_pneu)

class_weights = torch.tensor([w_normal, w_pneu], dtype=torch.float32).to(device)

class_weights

tensor([1.9441, 0.6731], device='cuda:0')

In [9]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total

In [11]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [12]:
num_epochs = 5
best_val_loss = float("inf")

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "models/efficientnet_best.pth")

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

Epoch [1/5]
Train Loss: 0.3249 | Train Acc: 0.8674
Val   Loss: 0.2359 | Val   Acc: 0.8669
----------------------------------------
Epoch [2/5]
Train Loss: 0.2173 | Train Acc: 0.9166
Val   Loss: 0.2038 | Val   Acc: 0.8889
----------------------------------------
Epoch [3/5]
Train Loss: 0.1908 | Train Acc: 0.9286
Val   Loss: 0.2234 | Val   Acc: 0.8736
----------------------------------------
Epoch [4/5]
Train Loss: 0.1761 | Train Acc: 0.9300
Val   Loss: 0.1948 | Val   Acc: 0.8898
----------------------------------------
Epoch [5/5]
Train Loss: 0.1694 | Train Acc: 0.9374
Val   Loss: 0.1760 | Val   Acc: 0.9033
----------------------------------------


In [13]:
model.load_state_dict(torch.load("models/efficientnet_best.pth"))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [14]:
import torch.nn.functional as F

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Test Accuracy: {correct/total:.4f}")

Test Accuracy: 0.8750


In [15]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))

              precision    recall  f1-score   support

      NORMAL       0.87      0.79      0.83       234
   PNEUMONIA       0.88      0.93      0.90       390

    accuracy                           0.88       624
   macro avg       0.87      0.86      0.86       624
weighted avg       0.87      0.88      0.87       624



In [17]:
import numpy as np

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1)

        all_probs.extend(probs[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())

thr = 0.6

preds_thr = (np.array(all_probs) >= thr).astype(int)

print(classification_report(all_labels, preds_thr, target_names=["NORMAL", "PNEUMONIA"]))

              precision    recall  f1-score   support

      NORMAL       0.84      0.83      0.84       234
   PNEUMONIA       0.90      0.91      0.90       390

    accuracy                           0.88       624
   macro avg       0.87      0.87      0.87       624
weighted avg       0.88      0.88      0.88       624



In [18]:
torch.save(model.state_dict(), "models/efficientnett_final.pth")